# Enterprise Gemini Agent Session Batch Transcription Pipeline

This notebook orchestrates the stateful, highly scalable batch transcription of VHF/UHF radio communications across hundreds of audio files. It combines **Google Cloud Vertex AI Agent Engine Sessions** for persistent conversational turn tracking with an advanced hybrid control plane that routes audio payloads natively over the web to custom multi-region fine-tuned models.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/gemini_agent_session/transcribe_masked_audio.ipynb)

---

## 🏛️ System Architecture & Innovations

### 1. Hybrid Control Plane (Single-Region Memory + Multi-Region ML Inference)
* **The Control Plane**: To utilize premium evaluation features and stable session logging, all conversational context history (`VertexAiSessionService`) is managed strictly within the single-region `us-central1` database.
* **The ML Compute Target**: When evaluating custom fine-tuned models (`6182316388472848384`), inference traffic is dynamically routed to the multi-region `us` serving infrastructure using our canonical `MultiRegionGemini` subclass. This immutably preserves custom Vertex AI client configurations across Pydantic compilation sweeps.

### 2. Pure Transparent Context Ownership (Zero Client State)
* **The Core Pattern**: In this pipeline, the multi-turn conversational memory graph is **100% delegated to and owned by the Cloud Agent Engine Control Plane (`VertexAiSessionService`)**.
* **Zero Client Boilerplate**: In our Colab execution lanes, we do not build, maintain, append, or convey raw conversational history arrays (`messages.append`). The context plane is entirely invisible and transparent to us in the notebook.
* **Maintainer Advisory**: Future maintainers and autonomous agents MUST NOT attempt to introduce client-side message history tracking or execute stateless non-conversational unary prediction fallbacks. We purely hand our raw `Content` packet and `session_id` pointer to `channel_runner.run_async()`, allowing the Cloud Control Plane to automatically maintain state and resolve tactical jargon across turns.

### 3. Standalone Parent Recording Sequential Lock
* VHF/UHF tactical radio transmissions are chunked across consecutive files (`seg000`, `seg001`, `seg002`). 
* To ensure the model retains perfect acoustic and acronym anchoring, we immutably group child segments under their parent audio recording folder name (`Path(audio_filepath).parent.name`) and execute them sequentially in strict chronological order `(offset, start_time)`. Independent parent audio recordings process concurrently, while child segments belonging to the same recording process as an indivisible sequential unit building on itself.

### 4. Pure Native GCS File Pointer Streaming (`Part.from_uri`)
* In our highly optimized enterprise architecture, all audio payloads are ingested natively as Google Cloud Storage pointers (`gs://...`) via `Part.from_uri()`. 
* This entirely eliminates legacy HTTPS Signed URL generation coroutines, completely bypassing local downloading and container RAM bloat while allowing the custom Vertex AI ML Endpoint to execute direct, ultra-fast VPC storage reads over Google Cloud's high-speed backbone.

### 5. Fail-Fast Execution Quality Gates
* Long-running concurrent batch jobs must be deeply protected against resource waste. 
* We inject rigorous Fail-Fast Exception Gates that break out of the standard 5-attempt retry loop immediately upon encountering fatal infrastructure errors (`403 PERMISSION_DENIED`, `404 NOT_FOUND`, `INVALID_ARGUMENT`), aborting doomed concurrent tasks instantly.

### 6. High-Performance Caching & GCS Sync
* Resilient batch lanes survive preemption seamlessly. 
* The pipeline automatically checkpoints intermediate transcription results directly to an NDJSON storage file, synchronizing with remote GCS storage buckets in background coroutine threads to naturally deduplicate and resume massive evaluations across preemption cycles.


In [ ]:
# @title Install dependencies
# Note: pip's dependency check may report benign advisory warnings regarding pre-installed Colab packages. These are standard environment quirks and can be safely ignored.
%pip install -q --upgrade \
    "google-adk" \
    "google-cloud-aiplatform" \
    "google-cloud-storage" \
    "google-genai>=2.3,<3" \
    loguru \
    tqdm

In [ ]:
# @title Imports
import asyncio
from collections import defaultdict
from datetime import timedelta
from functools import cached_property
import hashlib
import json
import os
from pathlib import Path
import random
import re
import subprocess
import sys
import time
from urllib.parse import urlparse

from google import adk, genai
from google.adk.events import Event
from google.adk.models import Gemini
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.runners import GetSessionConfig, RunConfig, Runner
from google.adk.sessions import VertexAiSessionService
from google.api_core import retry as api_retry
from google.api_core import retry_async as api_retry_async
from google.api_core.exceptions import GoogleAPICallError
from google.cloud import storage
from google.colab import auth, userdata
from google.genai import types
from google.genai.errors import ClientError
from google.oauth2 import credentials as oauth_creds
from IPython.display import display
from loguru import logger
import pandas as pd
from tqdm.auto import tqdm
import vertexai

In [ ]:
# @title Define constants and initial logging

# @markdown ### Model Selection
# @markdown **Option A: Select a base model**
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}

# @markdown **Option B: Use a Tuned ML Model (Deployed in multi-region us)**
USE_CUSTOM_MODEL = True  # @param {type:"boolean"}
# @markdown Enter just the numeric ID of your custom Target Endpoint in multi-region us (e.g., `6182316388472848384`):
CUSTOM_MODEL_ID = "6182316388472848384"  # @param {type:"string"}
GCP_CUSTOM_MODEL_NAME = "wd-radio-sft-2026-05-29-wd-internal-a16-gemini31-flash-lite"  # @param {type:"string"}

# @markdown ### Control Plane Session Configuration
# @markdown **How to find your active Reasoning Engine ID (AGENT_ENGINE_ID):**
# @markdown 1. Open the [Vertex AI Reasoning Engines Console](https://console.cloud.google.com/vertex-ai/reasoning-engines?project=automatic-hawk-481415-m9).
# @markdown 2. Look in the active deployments list for your agent (or run `gcloud beta ai reasoning-engines list --project=automatic-hawk-481415-m9`).
# @markdown 3. Copy the numeric **Reasoning Engine ID** (e.g., `6714487163142012928`) and paste it below:
AGENT_ENGINE_ID = "6714487163142012928"  # @param {type:"string"}

AGENT_NAME = "radio_transcript_session_agent"
APP_NAME = "contextual_audio_pipeline"
USER_ID = "radio_transcription_worker"

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCP_PROJECT_NUMBER = userdata.get("GCP_PROJECT_NUMBER")
GCS_BUCKET = userdata.get("GCS_BUCKET")

# @markdown ### GCP Infrastructure Configuration
# @markdown Active single-region for Agent Engine (used by base models):
GCP_LOCATION = "us-central1"  # @param {type:"string"}

# @markdown ### Input/Output Configuration
INPUT_AUDIO_DIR = "fire_notifications/eval_audio"  # @param {type:"string"}
OUTPUT_TRANSCRIPT_DIR = "fire_notifications/eval"  # @param {type:"string"}
EXPERIMENT_NAME = "fn_val_v1"  # @param {type:"string"}

AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
AUDIO_MASKING = True  # @param {type:"boolean"}

assert INPUT_AUDIO_DIR, "INPUT_AUDIO_DIR must be provided and cannot be empty."
assert OUTPUT_TRANSCRIPT_DIR, (
    "OUTPUT_TRANSCRIPT_DIR must be provided and cannot be empty."
)
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."
assert not (AUDIO_PREPROCESSING and AUDIO_MASKING), (
    "Cannot enable both AUDIO_PREPROCESSING and AUDIO_MASKING simultaneously."
)

GCS_INPUT_DIR = INPUT_AUDIO_DIR
if not (AUDIO_PREPROCESSING or AUDIO_MASKING):
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"
elif AUDIO_MASKING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_masked"

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in Colab userdata."
assert GCS_BUCKET, "GCS_BUCKET must be provided in Colab userdata."

# Pipeline Control
# fmt: off
OVERWRITE_EXISTING = False  # @param {type:"boolean"}
# fmt: on

# Streamlined Output Path Generation
if USE_CUSTOM_MODEL:
    assert CUSTOM_MODEL_ID, "CUSTOM_MODEL_ID must be provided."
    MODEL_ID_DIR = GCP_CUSTOM_MODEL_NAME
else:
    MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)

GCS_OUTPUT_BASE = (
    f"transcripts/{OUTPUT_TRANSCRIPT_DIR}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)

MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/segmented_audio/{GCS_INPUT_DIR}/batch_manifest.jsonl"
)
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"
CHECKPOINT_FILE = "interim_batch_checkpoints.ndjson"

SYSTEM_PROMPT = """
Evaluate all audio specifically as VHF/UHF fire-related dispatch radio traffic. The audio likely contains mic clicks, RF static, radio hum, and possibly some unintelligible speech. The speakers use heavy jargon.

EXPECTED TERMINOLOGY:
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, pitch branch alarm, automatic tactical engine copy unit code cancel go branch unit exposure.

CRITICAL RULES:
1. Output the transcript exactly as said, with no newlines.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. Format all unit identifiers as the unit type followed by digits (e.g., Engine 41, Battalion 2).
4. Do not continue the speech segment beyond what is spoken.
5. QUALITY GATE: Transcribe only what you hear with high acoustic certainty. If a portion of audio is obscured, noisy, or ambiguous, you MUST replace that specific portion with [UNINTELLIGIBLE]. Do not attempt to phonetically guess ambiguous noise.

TASK:
Transcribe the attached audio. Output strictly the transcript.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
    "thinking_config": types.ThinkingConfig(thinking_budget=0),
}

NUM_RECENT_EVENTS = 260
CONCURRENCY_LIMIT = 10
MAX_RETRIES = 5
SOCKET_TIMEOUT = 180.0

logger.remove()
logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level="WARNING"
);

In [ ]:
# @title Define MultiRegion ML Subclass
# Master MultiRegion Gemini Subclass Immutably Solving Multi-Region RunConfig Overwrites
class MultiRegionGemini(Gemini):
    @cached_property
    def api_client(self) -> genai.Client:
        logger.info(
            "MultiRegionGemini custom api_client executing! Connecting directly to standard region 'us' API gateway..."
        )
        return genai.Client(
            project=GCP_PROJECT_ID, location="us", vertexai=True
        )

In [ ]:
# @title Authenticate user and configure Google Cloud Session

auth.authenticate_user()

# CRITICAL SAFEGUARD: Ensure underlying gcloud CLI in the Colab container is perfectly synchronized with our target GCP Project
!gcloud config set project {GCP_PROJECT_ID} --quiet

logger.success(
    f"Successfully authenticated user and configured gcloud terminal specifically for {GCP_PROJECT_ID}."
)

In [ ]:
# @title Base Control Plane & Agent Instantiation

# Canonical Vertex Client & Target Model Initialization
storage_client = storage.Client(project=GCP_PROJECT_ID)
vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)
session_service = VertexAiSessionService(
    project=GCP_PROJECT_ID,
    location=GCP_LOCATION,
    agent_engine_id=AGENT_ENGINE_ID,
)
checkpoint_write_lock = asyncio.Lock()

if USE_CUSTOM_MODEL:
    TARGET_MODEL = MultiRegionGemini(
        model=f"projects/{GCP_PROJECT_NUMBER}/locations/us/endpoints/{CUSTOM_MODEL_ID}"
    )
else:
    TARGET_MODEL = MultiRegionGemini(
        model=f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}"
    )

audio_agent = adk.Agent(
    name=AGENT_NAME,
    model=TARGET_MODEL,
    instruction=SYSTEM_PROMPT.strip(),
    generate_content_config=types.GenerateContentConfig(
        temperature=GENERATION_CONFIG["temperature"],
        max_output_tokens=GENERATION_CONFIG["max_output_tokens"],
        thinking_config=GENERATION_CONFIG["thinking_config"],
        safety_settings=SAFETY_SETTINGS,
    ),
)

In [ ]:
# @title Pipeline Execution Logic

# Global Lock specifically to throttle Vertex Session Write Requests Quota endpoint
session_quota_lock = asyncio.Lock()

# Foundational enterprise AsyncIO Retry Policy protecting all streaming network channels
custom_async_retry = api_retry_async.AsyncRetry(
    predicate=api_retry.if_exception_type(
        (ClientError, GoogleAPICallError, asyncio.TimeoutError, ConnectionError)
    ),
    initial=1.0,
    maximum=60.0,
    multiplier=2.0,
    deadline=300.0,
)


def get_gcs_checkpoint_blob():
    """Derives GCS checkpoint path consistently for both load and upload."""
    out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
    out_path = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
    checkpoint_blob_path = "/".join(out_path[:-1]) + "/" + CHECKPOINT_FILE
    return storage_client.bucket(out_bucket).blob(checkpoint_blob_path)


def load_gcs_checkpoint() -> dict[str, str]:
    blob = get_gcs_checkpoint_blob()
    records = {}

    if blob.exists():
        logger.info(
            f"Found existing checkpoint on GCS: {blob.name}. Loading..."
        )
        blob.download_to_filename(CHECKPOINT_FILE)

        # 1. Read all records, mapping by audio_filepath to naturally deduplicate
        with open(CHECKPOINT_FILE, "r") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if not record.get("error"):
                        records[record["audio_filepath"]] = record

        # 2. Rewrite the local checkpoint file to be clean of errors and duplicates
        with open(CHECKPOINT_FILE, "w") as f:
            for record in records.values():
                f.write(json.dumps(record) + "\n")

        logger.info(f"Loaded {len(records)} completed records from GCS.")
    else:
        logger.info("No remote checkpoint found. Starting fresh.")

    # 3. Return the mapping expected by the rest of the notebook
    return {filepath: rec["transcript"] for filepath, rec in records.items()}


async def upload_checkpoint_to_gcs() -> None:
    """Synchronizes the local checkpoint file to GCS safely using CPython thread offloading."""
    async with checkpoint_write_lock:
        if os.path.exists(CHECKPOINT_FILE):
            try:
                blob = get_gcs_checkpoint_blob()
                await asyncio.to_thread(
                    blob.upload_from_filename, CHECKPOINT_FILE
                )
            except Exception as sync_err:
                logger.warning(
                    f"Failed to backup checkpoint to GCS: {sync_err}"
                )


async def process_single_channel(
    channel_id: str,
    segment_entries: list[dict],
    completed_records: dict[str, str],
    semaphore: asyncio.Semaphore,
    pbar: tqdm,
) -> list[dict]:
    """Processes a channel using Vertex AI Agent Engine sessions to automatically maintain context."""
    results = []

    # ============================================================================================
    # 🏛️ SENIOR ARCHITECTURAL SPECIFICATION: PURE TRANSPARENT CONTEXT OWNERSHIP & SEQUENTIAL LOCK
    # 1. Zero Client State: The multi-turn conversational context is 100% owned and managed by the
    #    Cloud Agent Engine Session (`session_id`). Do not build or convey message arrays here.
    # 2. Sequential Unit Lock: Child segments (`seg000`, `seg001`) belonging to this parent recording
    #    MUST be processed sequentially in strict chronological order building on each other.
    #    DO NOT execute stateless or independent non-conversational unary prediction fallbacks.
    # 3. Simplicity First & Context Purity: Completely avoid secondary duplicate turn recovery
    #    attempts. Rely purely on foundational upstream AsyncIO/Pydantic backoff to entirely protect
    #    the deployed session memory graph from duplicate turn pollution and broken recitations.
    # ============================================================================================

    async with semaphore:
        completed_in_channel = sum(
            1
            for e in segment_entries
            if e["audio_filepath"] in completed_records
        )
        logger.info(
            f"Starting agent session for {channel_id} ({len(segment_entries)} total segments, {completed_in_channel} already cached)"
        )

        channel_user_id = f"{USER_ID}_{channel_id}"
        session_id = None
        new_session = None

        for retry in range(MAX_RETRIES):
            try:
                # 100% Fully staggered, rate-limited session creation to completely defeat 429 Quota errors
                async with session_quota_lock:
                    new_session = await session_service.create_session(
                        user_id=channel_user_id,
                        app_name=AGENT_ENGINE_ID,
                        display_name=channel_id,
                    )
                    await asyncio.sleep(0.5)  # Minimal 500ms Quota stagger
                if not new_session or not new_session.id:
                    raise ValueError(f"Invalid session created: {new_session}")
                session_id = new_session.id
                break
            except Exception as e:
                logger.warning(
                    f"Attempt {retry + 1} failed to create session for {channel_id}: {e}"
                )
                if isinstance(e, (ClientError, GoogleAPICallError)) and (
                    "403" in str(e)
                    or "404" in str(e)
                    or "PERMISSION_DENIED" in str(e)
                    or "NOT_FOUND" in str(e)
                ):
                    logger.error(
                        f"🚨 FATAL SESSION ERROR: {e}. Aborting pipeline immediately."
                    )
                    raise RuntimeError(
                        f"Pipeline aborted due to fatal unrecoverable session error: {e}"
                    ) from e

                # Jittered exponential backoff to completely defeat Thundering Herd
                jitter_sleep = (2**retry) + random.uniform(1.0, 3.0)
                await asyncio.sleep(jitter_sleep)

        if not session_id or not new_session:
            error_msg = f"Failed to create Vertex AI session for channel {channel_id} after {MAX_RETRIES} retries."
            logger.error(error_msg)
            for entry in segment_entries:
                results.append(
                    {
                        "example_id": channel_id,
                        "audio_filepath": entry["audio_filepath"],
                        "transcript": None,
                        "error": error_msg,
                    }
                )
                pbar.update(1)
            return results

        # Fully completely isolate the concurrent ADK Runner instance to ensure 100% airtight asyncio thread safety!
        channel_runner = Runner(
            agent=audio_agent,
            app_name=AGENT_ENGINE_ID,
            session_service=session_service,
        )

        run_config = RunConfig(
            get_session_config=GetSessionConfig(
                create_if_not_exists=False, num_recent_events=NUM_RECENT_EVENTS
            )
        )

        try:
            # 🌿 Process exactly sequentially in strict chronological order without a split!
            for turn_index, entry in enumerate(segment_entries, start=1):
                uri = entry["audio_filepath"]

                if uri in completed_records:
                    cached_transcript = completed_records[uri]
                    logger.info(
                        f"[CACHE HIT] Restored transcript for {uri} (Turn {turn_index}/{len(segment_entries)})"
                    )
                    results.append(
                        {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": cached_transcript,
                            "error": None,
                        }
                    )

                    pbar.update(1)
                    continue

                # PURE NATIVE GCS FILE POINTER STREAMING (Zero Signed URLs!)
                audio_part = types.Part.from_uri(
                    file_uri=uri, mime_type="audio/flac"
                )

                message = types.Content(role="user", parts=[audio_part])

                final_error_msg = "Unknown error"

                # 🌿 1. Upstream retried coroutine beautifully accumulating ALL intermediate content packets
                @custom_async_retry
                async def execute_streaming_turn():
                    streamed_text_chunks = []
                    finish_reason = None
                    embedded_error_msg = None

                    async for event in channel_runner.run_async(
                        user_id=channel_user_id,
                        session_id=session_id,
                        new_message=message,
                        run_config=run_config,
                    ):
                        if (
                            hasattr(event, "error_message")
                            and event.error_message
                        ):
                            embedded_error_msg = event.error_message
                        if (
                            hasattr(event, "finish_reason")
                            and event.finish_reason
                        ):
                            finish_reason = event.finish_reason

                        if getattr(event, "content", None) and getattr(
                            event.content, "parts", None
                        ):
                            for part in event.content.parts:
                                if hasattr(part, "text") and part.text:
                                    streamed_text_chunks.append(part.text)

                        if event.is_final_response():
                            if (
                                hasattr(event, "raw_response")
                                and event.raw_response.candidates
                            ):
                                finish_reason = (
                                    finish_reason
                                    or event.raw_response.candidates[
                                        0
                                    ].finish_reason
                                )

                    # 🛡️ Highly rigorous Enum string filter entirely outsmarting FinishReason Protobuf wrappers
                    if embedded_error_msg or (
                        finish_reason
                        and "STOP"
                        not in str(
                            getattr(finish_reason, "name", finish_reason)
                        )
                        and "UNSPECIFIED"
                        not in str(
                            getattr(finish_reason, "name", finish_reason)
                        )
                        and finish_reason != 0
                    ):
                        logger.warning(
                            f"[API STREAM WARNING] {uri}: {embedded_error_msg or 'No embedded message'} (Reason: {finish_reason})"
                        )

                    full_text = "".join(streamed_text_chunks).strip()
                    # 🌿 Simplicity First Static Gate: flawlessly accepts an empty string ("") as a highly accurate return
                    if not embedded_error_msg and (
                        finish_reason is None
                        or "STOP"
                        in str(getattr(finish_reason, "name", finish_reason))
                        or "UNSPECIFIED"
                        in str(getattr(finish_reason, "name", finish_reason))
                        or finish_reason == 0
                    ):
                        return full_text

                    return full_text if full_text else None

                logger.debug(
                    f"[API CALL] Sending streaming request for {uri} in session {session_id}..."
                )
                req_start_time = time.time()
                try:
                    transcript = await asyncio.wait_for(
                        execute_streaming_turn(), timeout=SOCKET_TIMEOUT
                    )
                except Exception as stream_err:
                    transcript = None
                    final_error_msg = f"Streaming lane failed after upstream backoff: {stream_err}"
                    logger.warning(
                        f"[API STREAM FAILED] {uri}: {final_error_msg}"
                    )

                req_duration = time.time() - req_start_time

                # 🌿 2. Pure Execution Quality Gate (Zero Speculative Secondary Hybrid Context Pollution!)
                if transcript is not None:
                    if transcript == "":
                        logger.success(
                            f"[API SUCCESS] Pure ambient static/silence confirmed for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                        )
                    else:
                        logger.success(
                            f"[API SUCCESS] Final transcript acquired for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                        )
                    result_dict = {
                        "example_id": channel_id,
                        "audio_filepath": uri,
                        "transcript": transcript,
                        "error": None,
                    }
                    results.append(result_dict)

                    async with checkpoint_write_lock:
                        with open(CHECKPOINT_FILE, "a") as f:
                            f.write(json.dumps(result_dict) + "\n")
                else:
                    logger.error(
                        f"[API ERROR] {uri} - Final failure: {final_error_msg}"
                    )
                    results.append(
                        {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": None,
                            "error": final_error_msg,
                        }
                    )
                    async with checkpoint_write_lock:
                        with open(CHECKPOINT_FILE, "a") as f:
                            f.write(json.dumps(results[-1]) + "\n")

                pbar.update(1)

            try:
                await upload_checkpoint_to_gcs()
            except Exception as sync_err:
                logger.warning(
                    f"Failed to backup checkpoint to GCS for channel {channel_id}: {sync_err}"
                )

        finally:
            if session_id:
                logger.info(
                    f"Purging session {session_id} for channel {channel_id}..."
                )
                for purge_retry in range(MAX_RETRIES):
                    try:
                        async with session_quota_lock:
                            await session_service.delete_session(
                                session_id=session_id,
                                user_id=channel_user_id,
                                app_name=AGENT_ENGINE_ID,
                            )
                            await asyncio.sleep(
                                0.2
                            )  # Gentle 200ms purge stagger
                        break
                    except Exception as purge_err:
                        logger.warning(
                            f"Purge attempt {purge_retry + 1} failed for session {session_id}: {purge_err}"
                        )
                        await asyncio.sleep(
                            (2**purge_retry) + random.uniform(0.5, 1.5)
                        )

    return results


async def main() -> None:
    completed_records = {}

    if OVERWRITE_EXISTING:
        logger.info("OVERWRITE_EXISTING is True. Wiping outputs.")
        out_bucket_name = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[
            0
        ]
        out_blob_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        out_blob = storage_client.bucket(out_bucket_name).blob(out_blob_path)
        if out_blob.exists():
            out_blob.delete()
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
    else:
        logger.info(
            "OVERWRITE_EXISTING is False. Resuming from GCS checkpoint..."
        )
        completed_records = load_gcs_checkpoint()

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not manifest_blob.exists():
        raise FileNotFoundError(f"Manifest not found at {MANIFEST_URI}")

    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)
    total_segments = 0

    # 🌿 100% FOOLPROOF MASTER GROUPING: Immutably groups all child slices under their standalone Parent Audio Recording unit folder name!
    for line in content:
        if line.strip():
            entry = json.loads(line)
            parent_audio_unit = Path(entry["audio_filepath"]).parent.name
            entry["example_id"] = parent_audio_unit
            channels[parent_audio_unit].append(entry)
            total_segments += 1

    # 🌿 Highly rigorous multi-dimensional chronological sort guaranteeing zero out-of-order segment splits!
    for ch in channels:

        def get_sort_key(x):
            return (
                x.get("offset", 0),
                x.get("start_time", 0),
                x.get("audio_filepath", ""),
            )

        channels[ch].sort(key=get_sort_key)

    active_channels = {}
    missing_total = 0
    for cid, entries in channels.items():
        missing = [
            e for e in entries if e["audio_filepath"] not in completed_records
        ]
        if missing:
            active_channels[cid] = entries
            missing_total += len(missing)

    if not active_channels:
        logger.info("Everything complete.")
    else:
        active_segments_count = sum(
            len(entries) for entries in active_channels.values()
        )
        logger.info(
            f"Resuming {len(active_channels)} active independent audio recording units. Processing {missing_total} pending segments."
        )

        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        with tqdm(
            total=active_segments_count, desc="Processing Transcriptions"
        ) as pbar:
            tasks = [
                process_single_channel(
                    cid, entries, completed_records, semaphore, pbar
                )
                for cid, entries in active_channels.items()
            ]
            await asyncio.gather(*tasks)

    # Final Summary and GCS Upload
    final_successes = {}
    final_errors = []

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            lines = f.readlines()

        final_ndjson = "".join(lines)
        out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            final_ndjson
        )
        logger.info(f"Results saved to {CONSISTENT_OUTPUT_URI}")

        for line in lines:
            if line.strip():
                record = json.loads(line)
                if record.get("error"):
                    final_errors.append(record)
                else:
                    final_successes[record["audio_filepath"]] = record[
                        "transcript"
                    ]

    logger.info(
        f"Pipeline Complete. Total Successful Transcripts: {len(final_successes)}"
    )

    if final_errors:
        logger.error(
            f"\u26a0\ufe0f WARNING: {len(final_errors)} segments failed! Please re-run the pipeline cell to retry the failures."
        )
    else:
        logger.success("\ud83c\udf89 All segments processed successfully!")

    if final_successes:
        df = pd.DataFrame(
            list(final_successes.items()),
            columns=["audio_filepath", "transcript"],
        )
        df["example_id"] = df["audio_filepath"].apply(
            lambda x: Path(x).parent.name
        )
        display(df[["example_id", "audio_filepath", "transcript"]].head(10))

In [ ]:
# @title Execute Batch Transcriptions
await main()

In [ ]:
# @title Inspect Historical Session Content (Verify User & Model Event Retention)
# Inspect the event history of the active sessions to confirm audio and transcript retention
logger.warning("Fetching active sessions from Vertex AI...")
sessions_resp = await session_service.list_sessions(app_name=APP_NAME)

if sessions_resp and sessions_resp.sessions:
    sample_session = sessions_resp.sessions[-1]  # Pick the latest session
    logger.warning(
        f"Retrieving session content for Session ID: {sample_session.id} (User: {sample_session.user_id})"
    )

    session_obj = await session_service.get_session(
        user_id=sample_session.user_id,
        app_name=APP_NAME,
        session_id=sample_session.id,
    )

    if session_obj and hasattr(session_obj, "events") and session_obj.events:
        print(
            f"--- Active Session Events Summary ({len(session_obj.events)} total events) ---"
        )
        for idx, evt in enumerate(
            session_obj.events[:10]
        ):  # Inspect first 10 turns
            role = evt.author
            ts = evt.timestamp

            if role == "model":
                text_part = (
                    evt.content.parts[0].text
                    if evt.content and evt.content.parts
                    else "None"
                )
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Transcript: {text_part}"
                )
            elif role == "user":
                uri = "Audio Clip"
                if (
                    evt.content
                    and evt.content.parts
                    and hasattr(evt.content.parts[0], "file_data")
                ):
                    uri = evt.content.parts[0].file_data.file_uri
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Audio File: {uri}"
                )

        if len(session_obj.events) > 10:
            print(
                f"... and {len(session_obj.events) - 10} more events alternating between User (Audio) and Model (Transcript)."
            )
    else:
        print("No events recorded in this session yet.")
else:
    print(
        "No active sessions found. (Note: Sessions are automatically purged at the end of channel processing)."
    )

In [ ]:
# @title Validate Pipeline Integrity
# Count lines in manifest
m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
manifest_content = (
    storage_client.bucket(m_bucket)
    .blob(m_path)
    .download_as_text()
    .strip()
    .split("\n")
)
expected_count = len([l for l in manifest_content if l.strip()])

# Count lines in output
o_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
o_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
output_content = (
    storage_client.bucket(o_bucket)
    .blob(o_path)
    .download_as_text()
    .strip()
    .split("\n")
)
actual_count = len([l for l in output_content if l.strip()])

print(f"--- Pipeline Validation ---")
print(f"Expected Segments (Manifest): {expected_count}")
print(f"Actual Transcripts (Output):   {actual_count}")

if expected_count == actual_count:
    print("\n✅ SUCCESS: All segments were transcribed and recorded.")
else:
    print(
        f"\n❌ WARNING: Mismatch detected! Missing {expected_count - actual_count} segments."
    )

In [ ]:
# @title Session Isolation & Channel ID Audit


def audit_manifest_isolation():
    print(f"--- Auditing Manifest: {MANIFEST_URI} ---\n")

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    content = (
        storage_client.bucket(m_bucket)
        .blob(m_path)
        .download_as_text()
        .strip()
        .split("\n")
    )

    channel_to_paths = defaultdict(set)
    path_to_channels = defaultdict(set)

    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        cid = entry.get("example_id")
        path = entry.get("audio_filepath")

        # Extract the source folder name from the path as a 'ground truth' source
        source_folder = path.split("/")[-2] if "/" in path else "unknown"

        channel_to_paths[cid].add(source_folder)
        path_to_channels[source_folder].add(cid)

    # Check 1: Does one channel ID map to multiple physical folders? (Session Hijacking)
    overlap_found = False
    for cid, sources in channel_to_paths.items():
        if len(sources) > 1:
            print(
                f"⚠️ COLLISION: Channel ID '{cid}' is being shared by multiple sources: {sources}"
            )
            print("   This WILL cause session hijacking and context pollution.")
            overlap_found = True

    # Check 2: Does one physical folder have multiple channel IDs?
    for source, cids in path_to_channels.items():
        if len(cids) > 1:
            print(
                f"ℹ️ Note: Source '{source}' is split across multiple IDs: {cids}"
            )

    if not overlap_found:
        print(
            "✅ Isolation Audit Passed: Each Channel ID maps to exactly one source directory."
        )

    print(f"\nUnique Sessions to be created: {len(channel_to_paths)}")


audit_manifest_isolation()